## Tardis Model

### 1. Package Importation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
cleaned_df = df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### 1.1 Analysis on a global basis

In [ ]:
#df = df.select_dtypes(exclude=['object'])
df = pd.get_dummies(df, columns=["Gare de départ", "Gare d'arrivée", "Service","Commentaire annulations","Commentaire retards à l'arrivée","Commentaire retards au départ"], drop_first=True)

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")

df["Annee"] = df["Date"].dt.year
df["Mois"] = df["Date"].dt.month

df = df.drop("Date", axis=1)


#df = pd.get_dummies(df, columns=["Gare de départ", "Gare d'arrivée", "Service"], drop_first=True)

X = df.drop("Retard moyen de tous les trains à l'arrivée",axis=1)
y = df["Retard moyen de tous les trains à l'arrivée"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
pipe = make_pipeline(StandardScaler(),LinearRegression())
pipe.fit(X_train, y_train) 


predictions = pipe.predict(X_test)   


print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R² Score:", r2_score(y_test, predictions))    

In [ ]:
import joblib
joblib.dump(pipe, 'model.pkl') 

In [ ]:
df_test = pd.DataFrame({
    "Gare de départ": ["LE MANS"],
    "Gare d'arrivée": ["PARIS MONTPARNASSE"],
    "Service": ["National"],
    "Date": ["2027-08"],
    "Durée moyenne du trajet": [None],
    "Nombre de circulations prévues": [None],
    "Nombre de trains annulés": [None],
    "Commentaire annulations": [None],
    "Nombre de trains en retard au départ": [None],
    "Retard moyen des trains en retard au départ": [None],
    "Retard moyen de tous les trains au départ": [None],
    "Commentaire retards au départ": [None],
    "Nombre de trains en retard à l'arrivée": [None],
    "Retard moyen des trains en retard à l'arrivée": [None],
    "Commentaire retards à l'arrivée": [None],
    "Nombre trains en retard > 15min": [None],
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)": [None],
    "Nombre trains en retard > 30min": [None],
    "Nombre trains en retard > 60min": [None],
    "Prct retard pour causes externes": [None],
    "Prct retard pour cause infrastructure": [None],
    "Prct retard pour cause gestion trafic": [None],
    "Prct retard pour cause matériel roulant": [None],
    "Prct retard pour cause gestion en gare et réutilisation de matériel": [None],
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)": [None]
})


def search_similate(cdf,df):

    GD = df["Gare de départ"].iloc[0]
    GA = df["Gare d'arrivée"].iloc[0]
    S = df["Service"].iloc[0]
    Date = df["Date"].iloc[0]
    
    target_date = datetime.strptime(Date, "%Y-%m")
    year = target_date.year
    month = target_date.month

    df["Annee"] = year
    df["Mois"] = month
    
    cdf["Date"] = pd.to_datetime(cdf["Date"], format="%Y-%m")
    
    cdf["Annee"] = cdf["Date"].dt.year
    cdf["Mois"] = cdf["Date"].dt.month
    
    filters = [
        (cdf["Gare d'arrivée"] == GA),
        (cdf["Mois"] == month),
        (cdf["Gare de départ"] == GD),
        (cdf["Annee"] == year),
        (cdf["Service"] == S)
    ]
    
    for i in range(len(filters), 0, -1):
        mask = filters[:i][0]
        for filter in filters[:i][1:]:
            mask = mask & filter        
        result = cdf[mask]  
        print(result)      
        if not result.empty:
            for col in df.columns:
                if pd.isna(df[col].iloc[0]) and col in result.columns:
                    df.at[0, col] = result.iloc[0][col]
            return True
    
    print("Aucun match trouvé")
    return False


search_similate(
    cleaned_df,
    df_test
)



df_test = pd.get_dummies(
    df_test,
    columns=[
        "Gare de départ",
        "Gare d'arrivée",
        "Service",
        "Commentaire annulations",
        "Commentaire retards à l'arrivée",
        "Commentaire retards au départ"
    ],
    drop_first=True
)

df_test = df_test.drop("Date", axis=1)

df_test = df_test.reindex(columns=X.columns, fill_value=0)

predictions = pipe.predict(df_test)

print( "predictions : ",predictions[0])
